# HumanML3D 的263维特征向量都是什么？怎么验证？

# 省流模式：这里有一份最权威的 **HumanML3D 263维特征定义表**。

### 核心结论：前 4 维定义 (indices 0-3)

代码证据在 `process_file` 的末尾部分：
```python
# 'r_velocity' is (seq_len-1, 1) rotation velocity along y-axis
r_velocity = np.arcsin(r_velocity[:, 2:3]) 

# 'l_velocity' is (seq_len-1, 2) linear velovity on xz plane
l_velocity = velocity[:, [0, 2]]  # 所有帧的

# 'root_y'
root_y = positions[:, 0, 1:2]   # 所有帧的第0个joint的y值坐标，应该是[nframes，1，1]

# Concatenation
root_data = np.concatenate([r_velocity, l_velocity, root_y[:-1]], axis=-1)
```

**✅ 确定的顺序是：**
1.  **第 0 维**: **根节点 Y 轴角速度 (Root Angular Velocity)**。物理含义是“转弯速度”。
2.  **第 1 维**: **根节点 X 轴线速度 (Root Linear Velocity X, Local)**。物理含义是“左右平移/侧向速度”。
3.  **第 2 维**: **根节点 Z 轴线速度 (Root Linear Velocity Z, Local)**。物理含义是“前后移动速度”。
4.  **第 3 维**: **根节点 Y 轴高度 (Root Height)**。

这与你之前的理解（VelY角速, VelX, VelZ, PosY）是**完全一致**的。

---

### 🔍 263 维全貌考证 (The Full Spectrum)

HumanML3D 的特征是为了 Transformer 训练优化的，它包含了大量的**冗余信息**（Redundancy）。

| 维度范围 (Indices) | 维度数量 | 物理含义 (Semantics) | 代码来源证据 |
| :--- | :--- | :--- | :--- |
| **0 ~ 3** | **4** | **根节点基础信息**<br>0: 旋转角速度 ($r\_vel$)<br>1: X轴线速度 ($v\_x$)<br>2: Z轴线速度 ($v\_z$)<br>3: 地面高度 ($h$) | `root_data` |
| **4 ~ 66** | **63** | **局部关节位置 (Local Joint Positions)**<br>共21个关节（除去根节点），每个关节3维 (x,y,z)。<br>这是相对于根节点的相对坐标。 | `ric_data`<br>`(joints_num-1)*3` |
| **67 ~ 192** | **126** | **局部关节旋转 (Local Joint Rotations)**<br>共21个关节（除去根节点），每个关节6维。<br>使用的是 **6D Continuous Rotation** 表示法。 | `rot_data`<br>`(joints_num-1)*6` |
| **193 ~ 258** | **66** | **关节局部速度 (Joint Velocities)**<br>共22个关节（**包含**根节点），每个关节3维。<br>这是每一帧关节位置的差分。 | `local_vel`<br>`joints_num*3` |
| **259 ~ 262** | **4** | **脚接触标签 (Foot Contacts)**<br>左脚跟, 左脚尖, 右脚跟, 右脚尖。<br>(二值化或概率值) | `feet_l, feet_r` |

**总计验证**：$4 + 63 + 126 + 66 + 4 = 263$。**得证。**

---

### 💡 对你“旋转增强”代码的启示

基于这个考证，你之前的 `augment_content_rotation` 函数逻辑需要再次确认：

1.  **Dimensions 0-3 (Root)**:
    *   **第 0 维 (角速度)**: **不需要旋转**。因为你是绕 Y 轴整体旋转，转弯的相对角速度是不变的。
    *   **第 1, 2 维 (线速度)**: **需要旋转**。这正是把“向前走”变成“侧着走”的关键。
    *   **第 3 维 (高度)**: **不需要旋转**。

2.  **Dimensions 4-66 (Local Pos)**:
    *   **需要旋转**。这是相对根节点的 $(x, y, z)$，绕 Y 轴旋转会改变 $x$ 和 $z$。

3.  **Dimensions 67-192 (Local Rot 6D)**:
    *   **不需要旋转**（通常）。
    *   **深度解析**：这里的旋转是关节相对于父关节的局部旋转。如果你整体旋转了人的朝向（Root），**手臂相对于身体的姿态是不变的**。所以这 126 维不需要动。这大大简化了你的工作！

4.  **Dimensions 193-258 (Joint Vel)**:
    *   **需要旋转**。速度向量的方向会随身体朝向改变。

5.  **Dimensions 259-262 (Contacts)**:
    *   **不需要旋转**。脚有没有踩地跟朝向无关。

### ✅ 最终结论：你的增强逻辑是对的

你之前写的 `augment_content_rotation` 函数里：
*   旋转了 `features[..., 1:3]` (Root Vel) —— **正确**。
*   旋转了 `features[..., 4:70]` (Joint Pos) —— **正确**。
*   旋转了 `features[..., 70:136]` (Joint Vel? **Wait!**)

**🚨 这里的索引可能有误，请核对：**
*   **0-3**: Root (4)
*   **4-66**: Joint Pos (63) $\rightarrow$ 索引范围 `4 : 67`
*   **67-192**: Joint Rot (126) $\rightarrow$ 索引范围 `67 : 193` (**这些是旋转参数，不需要动！**)
*   **193-258**: Joint Vel (66) $\rightarrow$ 索引范围 `193 : 259` (**这些需要动！**)

**⚠️ 你的代码需要修正索引！**
你之前的代码好像是紧接着处理了，但根据特征表，中间隔着 **126 维的旋转特征**。

**修正后的增强代码逻辑**：

```python
    def augment_content_rotation(self, features):
        # ... (生成 c, s) ...
        
        # 1. 旋转 Root Linear Velocity (Indices 1, 2)
        # ... (保持不变) ...
        
        # 2. 旋转 Local Joint Positions (Indices 4 ~ 67)
        # 63 dims = 21 joints * 3
        joint_pos = features[..., 4:67].reshape(bs, frames, 21, 3) 
        # ... (执行旋转) ...
        features[..., 4:67] = joint_pos.reshape(bs, frames, 63)
        
        # 3. 跳过 Joint Rotations (Indices 67 ~ 193) !!!
        # 这些是身体内部的相对姿态，不需要动。
        
        # 4. 旋转 Local Joint Velocities (Indices 193 ~ 259)
        # 66 dims = 22 joints * 3
        joint_vel = features[..., 193:259].reshape(bs, frames, 22, 3)
        # ... (执行旋转) ...
        features[..., 193:259] = joint_vel.reshape(bs, frames, 66)
        
        return features
```

**一定要改索引！** 否则你会把“关节旋转参数”当成“关节速度”去旋转，那样数据就彻底乱了，模型绝对会崩。

# 模型这一轮的更改
好的，我们来系统地盘点一下，这一轮新的训练相较于原始的 MCM-LDM，你做了哪些**深刻且关键的改动**。

总的来说，你把一个容易“走捷径”的模型，改造成了一个**被迫学习解耦（Disentanglement）**的鲁棒模型。

---

### 全部改动清单 (All Modifications)

#### 1. 训练策略核心改动 (Training Strategy)

*   **引入旋转数据增强 (Rotation Augmentation)**:
    *   **改动**：在 `train_diffusion_forward` 中，对作为 Content 条件的动作（`feats_content`）应用了**随机 Y 轴旋转**。
    *   **目的**：**人为制造 Content 方向与 Trajectory 方向的不匹配**。这是本次修改的**灵魂**，旨在打破模型对 Content 轨迹的依赖，强迫它学习“姿态重定向 (Pose Re-orientation)”。

*   **强化条件丢弃 (Condition Dropout)**:
    *   **改动**：在训练时，以更高的概率（例如 50% 或更高）将 Content 条件（`cond_emb`）置为零。
    *   **目的**：在部分训练步中，让模型完全“看不见”Content，只能依赖 Trajectory 来生成动作。这进一步强化了 Trajectory 分支的权重，防止其在训练中“退化”。

#### 2. 数据流与特征处理改动 (Data Flow & Feature Processing)

*   **彻底的根节点屏蔽 (Root Masking)**:
    *   **改动**：在 `train_diffusion_forward` 中，将 `feats_content` 的前 4 维特征（包含角速度、线速度和高度）全部置零。
    *   **目的**：作为“双重保险”，彻底消除 Content 输入中的任何全局位移线索，让它变成一个纯粹的“原地姿态”提供者。

*   **强化轨迹编码器 (Trajectory Encoder Enhancement)**:
    *   **改动**：重写了 `TransEncoder`，使其**不再对时序信息进行池化（Pooling）**，而是输出与 Latent 长度对齐的**特征序列**。
    *   **目的**：让 Trajectory 条件不再是一个模糊的“全局偏移”信号，而是一个包含详细路径变化的**逐帧（Frame-wise）引导**。

#### 3. 模型架构微调 (Model Architecture)

*   **强力条件注入 (Stronger Conditioning Injection)**:
    *   **改动**：在 `MldDenoiser` 的 `forward` 方法中，将 Trajectory 特征序列与 Content 特征序列、噪声 Latent 序列进行**拼接（Concatenation）**，共同作为 DiT (Diffusion Transformer) 的输入。
    *   **目的**：将 Trajectory 从原来的“旁路调制（AdaLN）”提升为“主干输入”，极大地增强了其在注意力计算中的话语权，解决了“喧宾夺主”的问题。

*   **DiT Block 适配**:
    *   **改动**：修改了 `DiTBlock` 的 `forward` 函数，使其不再接收 Trajectory 作为 AdaLN 的输入，而是让 Style+Time 条件统一控制所有的调制参数（MSA 和 MLP）。
    *   **目的**：适配上述“强力注入”的改动，使架构逻辑自洽。

#### 4. (推理阶段) 新增功能 (Inference)

*   **空间引导模块 (Spatial Guidance Module)**:
    *   **改动**：新增了 `compute_spatial_guidance` 函数，实现了类似 OmniControl 的推理时优化。
    *   **目的**：通过在 Latent Space 进行梯度下降，**强制**生成的动作轨迹精确贴合给定的目标路径（如圆形、直线），作为最终效果的“兜底保障”。

*   **自定义轨迹生成器**:
    *   **改动**：新增了 `generate_custom_trajectory` 函数，用于在推理时生成标准化的、可控的轨迹（如圆形、直线）。
    *   **目的**：方便进行定量的、可视化的解耦效果验证。

---

### 总结：从“依赖”到“解耦”的蜕变

*   **原始 MCM-LDM**：是一个**强依赖 Content** 的模型。它默认 Content 和 Trajectory 是一致的，因此倾向于从信息更丰富的 Content 中学习一切，导致 Trajectory 分支被“架空”。

*   **你的新模型**：通过**旋转增强**制造冲突，通过**强注入**和**强 Dropout** 提升 Trajectory 话语权，最终通过**空间引导**确保精度。你构建了一个**真正意义上的多条件解耦模型**，其中：
    *   **Content 提供“怎么动”（姿态内核）**
    *   **Style 提供“什么感觉”（风格）**
    *   **Trajectory 提供“去哪里”（路径）**

这已经不仅仅是简单的 Debug，而是一次成功的**模型重构（Model Refactoring）**。